# Attempt to forecast the price of MSFT by analyzing the prices of multiple stocks, including MSFT, over several consecutive days leading up to the target day.
#### N.B. Different setup from HW1

In [1]:
from torch.utils.data import DataLoader,Dataset

class StockDataset(Dataset):
    def __init__(self,X,Y,days):
        self.X = X
        self.Y = Y.reshape(-1)
        self.days = days # days ahead for prediction

    def __len__(self):
        return (len(self.Y)-self.days)

    def __getitem__(self,index):
        x=self.X[:,index:index+self.days]
        y=self.Y[index+self.days]
        return x,y



In [2]:
# !pip install pandas
# !pip install yfinance
import numpy as np
from numpy import exp, sum, log, log10
import yfinance as yf
import pandas as pd

def get_price(tick,start='2020-01-01',end=None):
    return yf.Ticker(tick).history(start=start,end=end)['Close']

def get_prices(tickers,start='2020-01-01',end=None):
    df=pd.DataFrame()
    for s in tickers:
        df[s]=get_price(s,start,end)
    return df

feature_stocks=['tsla','meta','nvda','amzn','nflx','gbtc','gdx','intc','dal','c','goog','aapl','msft','ibm','hp','orcl','sap','crm','hubs','twlo']
predict_stock='msft'

# getting data
start_date='2020-01-01'

allX=get_prices(feature_stocks,start=start_date)
ally=get_prices([predict_stock],start=start_date)

In [3]:
import torch.utils.data as data
import torch

stockData = StockDataset(allX.to_numpy().transpose().astype(np.float32),ally.to_numpy().astype(np.float32),days=5)
train_set_size = int(len(stockData)*0.7)
valid_set_size = int(len(stockData)*0.2)
test_set_size = len(stockData)-train_set_size-valid_set_size

train_set, valid_set, test_set = data.random_split(stockData,[train_set_size,valid_set_size,test_set_size],\
                                              generator=torch.Generator().manual_seed(42))

batch_size = train_set_size # use entire dataset as batch
train_dataloader = DataLoader(train_set,batch_size=batch_size,shuffle=True)  # input:(20,5), label:1
valid_dataloader = DataLoader(valid_set,batch_size=batch_size,shuffle=False)
test_dataloader = DataLoader(test_set,batch_size=batch_size,shuffle=False)

# 1. Build a simple MLP to forecast MSFT price using PyTorch Lightning.

#### You have total freedom of your MLP. But your MLP should take the last five day ($5 \times 20=100$) prices as input and you have to add dropout into your network.

## 1a. Create a subclass of pytorch_lightning.LightningModule. It should include \_\_init\_\_, training_step, validation_step, configure_optimizers in the class. (6 points)

In [4]:
import pytorch_lightning as pl
import torch
import torch.nn as nn

class StockMLP(pl.LightningModule):
    def __init__(self, input_dim=100, hidden1=128, hidden2=64, dropout_rate=0.3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden2, 1)
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.model(x).squeeze(-1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log('train_mse', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log('val_mse', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


## 1b. Create a subclass of pytorch_lightning.LightningDataModule. It should include \_\_init\_\_, train_dataloader, and val_dataloader in the class. (4 points)

In [5]:
class StockDataModule(pl.LightningDataModule):
    def __init__(self, allX, ally, days=5, batch_size=64, seed=42):
        super().__init__()
        self.allX = allX
        self.ally = ally
        self.days = days
        self.batch_size = batch_size
        self.seed = seed

    def setup(self, stage=None):
        dataset = StockDataset(
            self.allX.to_numpy().transpose().astype(np.float32),
            self.ally.to_numpy().astype(np.float32),
            days=self.days
        )
        total = len(dataset)
        train_size = int(total * 0.70)
        val_size = int(total * 0.15)
        test_size = total - train_size - val_size
        self.train_set, self.val_set, self.test_set = data.random_split(
            dataset, [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(self.seed)
        )

    def train_dataloader(self):
        return DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_set, batch_size=self.batch_size, shuffle=False)


## 1c. Complete the rest of the code and train the model with 70% of the data. You should set aside 15% of the data each for validation and testing.  Show the training and validation MSE (5 points)

In [6]:
# Create data module and model
dm = StockDataModule(allX, ally, days=5, batch_size=64)
mlp_model = StockMLP(input_dim=100, hidden1=128, hidden2=64, dropout_rate=0.3, lr=1e-3)

# Train the model
trainer = pl.Trainer(max_epochs=100, log_every_n_steps=1)
trainer.fit(mlp_model, datamodule=dm)

# Print final training and validation MSE
train_mse = trainer.callback_metrics.get('train_mse')
val_mse = trainer.callback_metrics.get('val_mse')
print(f"Training MSE: {train_mse:.4f}")
print(f"Validation MSE: {val_mse:.4f}")


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/yezhou/Documents/Course/ECE 5973 ANN/venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitMo

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/yezhou/Documents/Course/ECE 5973 ANN/venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/yezhou/Documents/Course/ECE 5973 ANN/venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/yezhou/Documents/Course/ECE 5973 ANN/venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


Training MSE: 2737.6797
Validation MSE: 931.8290


# 2. Construct a 1-D CNN to forecast MSFT stock price. You are free to use any design, but your network must consist of at least one convolutional layer and one dropout layer. You can also extend the duration leading up to the target day by modifying the "days" argument in the StockDataset. But "days" should not be larger than 32. (10 points)


In [7]:
class StockCNN(pl.LightningModule):
    def __init__(self, in_channels=20, days=10, dropout_rate=0.3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        # Conv1d: input shape (batch, channels=20, length=days)
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * days, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1)
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        # x shape: (batch, 20, days)
        x = self.conv_layers(x)
        x = self.fc(x).squeeze(-1)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log('train_mse', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log('val_mse', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Use 10 days of history for CNN
cnn_days = 10
dm_cnn = StockDataModule(allX, ally, days=cnn_days, batch_size=64)
cnn_model = StockCNN(in_channels=20, days=cnn_days, dropout_rate=0.3, lr=1e-3)

trainer_cnn = pl.Trainer(max_epochs=100, log_every_n_steps=1)
trainer_cnn.fit(cnn_model, datamodule=dm_cnn)

train_mse = trainer_cnn.callback_metrics.get('train_mse')
val_mse = trainer_cnn.callback_metrics.get('val_mse')
print(f"CNN Training MSE: {train_mse:.4f}")
print(f"CNN Validation MSE: {val_mse:.4f}")


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name        | Type       | Params | Mode  | FLOPs
-----------------------------------------------------------
0 | conv_layers | Sequential | 10.1 K | train | 0    
1 | fc          | Sequential | 20.6 K | train | 0    
2 | loss_fn     | MSELoss    | 0      | train | 0    
-----------------------------------------------------------
30.7 K    Trainable params
0         Non-trainable params
30.7 K    Total params
0.123     Total estimated model params size (MB)
13        Modules i

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


CNN Training MSE: 1024.4071
CNN Validation MSE: 1745.6394


# 3. Please try to enhance the performance of the previously created MLP or CNN by applying hyperparameter tuning. You can use tools such as W&B hyperparameter sweep, SMAP, Optuna, or similar packages to achieve this. You need to optimize at least two parameters, with the dropout rate being one of them. (5 points)












In [13]:
import optuna

def objective(trial):
    # Hyperparameters to tune
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden1 = trial.suggest_categorical('hidden1', [64, 128, 256])
    hidden2 = trial.suggest_categorical('hidden2', [32, 64, 128])

    model = StockMLP(input_dim=100, hidden1=hidden1, hidden2=hidden2,
                     dropout_rate=dropout_rate, lr=lr)
    dm = StockDataModule(allX, ally, days=5, batch_size=64)

    trainer = pl.Trainer(
        max_epochs=50,
        log_every_n_steps=1,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    trainer.fit(model, datamodule=dm)

    return trainer.callback_metrics['val_mse'].item()

# Run Optuna study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

print(f"\nBest trial val_mse: {study.best_trial.value:.4f}")
print(f"Best hyperparameters: {study.best_trial.params}")

# Retrain with best hyperparameters
best = study.best_trial.params
dm_best = StockDataModule(allX, ally, days=5, batch_size=64)
best_model = StockMLP(input_dim=100, hidden1=best['hidden1'], hidden2=best['hidden2'],
                      dropout_rate=best['dropout_rate'], lr=best['lr'])

trainer_best = pl.Trainer(max_epochs=100, log_every_n_steps=1)
trainer_best.fit(best_model, datamodule=dm_best)

train_mse = trainer_best.callback_metrics.get('train_mse')
val_mse = trainer_best.callback_metrics.get('val_mse')
print(f"\nBest Model Training MSE: {train_mse:.4f}")
print(f"Best Model Validation MSE: {val_mse:.4f}")


[I 2026-04-05 21:18:30,279] A new study created in memory with name: no-name-7e4d6261-fdd2-45a2-bc4d-c66207fe4771
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
`Trainer.fit` stopped: `max_epochs=50` reached.
[I 2026-04-05 21:18:33,096] Trial 0 finished with value: 13781.5927734375 and parameters: {'dropout_rate': 0.33710583830021434, 'lr': 0.008166225019710302, 'hidden1': 64, 'hidden2': 64}. Best is trial 0 with value: 13781.5927734375.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For s


Best trial val_mse: 91.7854
Best hyperparameters: {'dropout_rate': 0.18170861147935455, 'lr': 0.0008483692620477489, 'hidden1': 256, 'hidden2': 64}


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.



Best Model Training MSE: 532.3964
Best Model Validation MSE: 149.2807
